<a href="https://www.kaggle.com/code/morescope/speciesnet-validation?scriptVersionId=247508998" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Introduction
As of 2025-06-12, the volunteers at rangers.urbanrivers have added 59,351 observations.  
These observations are not error-proof, but they are definitely a useful means of creating base-truth casses for image labeling.  

We have run speciesnet on all 200k+ images in the database and condensed the results to MongoDB s aiResults. 
This notebook serves to determine the rates of detector confidence and human labels for false negative and false positive results


In [1]:
# Data Handling
import pandas as pd
from collections import defaultdict

# IO - getting files and images
from pymongo import MongoClient
from kaggle_secrets import UserSecretsClient
import requests
import json
import os
import urllib.parse

# For randomizing which images get downloaded
import random
from tqdm.auto import tqdm

print("==== Loaded Libraries ====")

==== Loaded Libraries ====


# Accessing observations (image labels) through pyMongo
This version uses pymongo (MongoClient) 

In [2]:
%%time
# Get the stored mongo uri secret
user_secrets = UserSecretsClient()
mongo_uri = user_secrets.get_secret("MONGO_PROD")

# Access the server
client = MongoClient(mongo_uri)
db = client['test']
collection = db['cameratrapmedias']

# Fetch documents with at least one speciesConsensus entry
def fetch_all_obs():
    query = {"speciesConsensus.0": {"$exists": True}, "aiResults.0": {"$exists": True}}  # at least one item
    projection = {
        "_id": 0,
        "mediaID": 1,
        "publicURL": 1,
        "speciesConsensus": 1,
        "aiResults": 1
    }
    all_obs = list(collection.find(query, projection))
    print(f"Retrieved {len(all_obs)} documents with speciesConsensus and aiResults.")
    return all_obs

# Try the fetch operation
try:
    print("===== Starting MongoDB Fetch =====")
    obs_json = fetch_all_obs()
except Exception as e:
    print(f"Error during fetch: {e}")

===== Starting MongoDB Fetch =====
Retrieved 60400 documents with speciesConsensus and aiResults.
CPU times: user 1.49 s, sys: 291 ms, total: 1.79 s
Wall time: 4.38 s


In [3]:
obs_json[:2]

[{'mediaID': 'c112813a5f3b9cec26f95fad982b8d09',
  'publicURL': 'https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0001.JPG',
  'speciesConsensus': [{'observationType': 'blank',
    'scientificName': None,
    'taxonID': None,
    'count': 1,
    'accepted': False,
    'observationCount': 1,
    '_id': ObjectId('678178f047e6f831f49b6ea5')}],
  'aiResults': [{'modelName': 'speciesnet/PyTorch/v4.0.1a',
    'runDate': '2025-05-24',
    'confBlank': 1.0,
    'confHuman': 0.0,
    'confAnimal': 0.0}]},
 {'mediaID': '0647380f2d59692f5b2b642312844e9f',
  'publicURL': 'https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0002.JPG',
  'speciesConsensus': [{'observationType': 'human',
    'scientificName': None,
    'taxonID': None,
    'count': 1,
    'accepted': False,
    'observationCount': 1,
    '_id': ObjectId('678178f447e6f831f49b6eb6')}],
  'aiResults': [{'modelName': 'speciesnet/PyTorch/v4.0.1a',
 

In [4]:
# See what observation Types we have
observation_types = set()

for item in obs_json:
    for obs in item.get('speciesConsensus', []):
        obs_type = obs.get('observationType')
        if obs_type is not None:
            observation_types.add(obs_type)

print(observation_types)

{'vehicle', 'human', 'animal', 'blank'}


In [5]:
# Flatten to dataframe
flat_rows = []

for item in obs_json:
    media_id = item.get('mediaID')
    url = item.get('publicURL')
    
    # Use first entry of speciesConsensus and aiResults
    consensus = item.get('speciesConsensus', [{}])[0]
    ai = item.get('aiResults', [{}])[0]
    
    flat_rows.append({
        'mediaID': media_id,
        'publicURL': url,
        'observationType': consensus.get('observationType'),
        'scientificName': consensus.get('scientificName'),
        'observationCount': consensus.get('observationCount'),
        'confBlank': ai.get('confBlank'),
        'confHuman': ai.get('confHuman'),
        'confAnimal': ai.get('confAnimal'),
    })

# Create a flat DataFrame
df = pd.DataFrame(flat_rows)
display(df.head())

# Save to CSV file
df.to_csv("observations_and_airesults.csv", index=False)
print("file saved to .csv")

,mediaID,publicURL,observationType,scientificName,observationCount,confBlank,confHuman,confAnimal
0,c112813a5f3b9cec26f95fad982b8d09,https://urbanriverrangers.s3.amazonaws.com/ima...,blank,None,1,1.00,0.00,0.0
1,0647380f2d59692f5b2b642312844e9f,https://urbanriverrangers.s3.amazonaws.com/ima...,human,None,1,0.36,0.64,0.0
2,0db73c6c1efb4968c04a47e418ebeefb,https://urbanriverrangers.s3.amazonaws.com/ima...,human,None,1,0.37,0.63,0.0
3,31fc53de29056b4dd8bc7b1804617f00,https://urbanriverrangers.s3.amazonaws.com/ima...,human,None,1,0.32,0.68,0.0
4,14664d764836c5fd9a38284dc6103527,https://urbanriverrangers.s3.amazonaws.com/ima...,human,None,1,0.30,0.70,0.0


file saved to .csv


In [6]:
# How many obs counts are there.
df['observationCount'].value_counts()

observationCount
1     42052
2     13325
3      3563
4      1063
5       248
6        98
7        34
8        12
9         4
10        1
Name: count, dtype: int64

## Process the returned JSON for the fields we need
We're looking for the `mediaID` (our primary key),  

What the species consensus from human labeling is,  

And what the aiResult Probabilities are.

In [7]:
%%time
from collections import defaultdict
import pandas as pd

# Your JSON data here (replace this with your actual data loading step)
data = obs_json  # Replace with your JSON list

# Prepare list for rows
rows = []

for item in data:
    consensus_entries = item.get('speciesConsensus', [])
    ai_result = item.get('aiResults', [{}])[0]  # Assume only one aiResult per mediaID
    
    if not consensus_entries or not ai_result:
        continue

    obs_type = consensus_entries[0].get('observationType')
    
    if obs_type == 'blank':
        group = 'blank'
    elif obs_type == 'human':
        group = 'human'
    elif obs_type == 'animal':
        group = 'animal'
    
    rows.append({
        'group': group,
        'confBlank': ai_result.get('confBlank', 0),
        'confHuman': ai_result.get('confHuman', 0),
        'confAnimal': ai_result.get('confAnimal', 0)
    })

# Convert to DataFrame
df = pd.DataFrame(rows)

# Show the distribution by group
distribution = df.groupby('group').agg(['mean', 'min', 'max'])
with pd.option_context('display.width', 0, 'display.max_colwidth', None):
    display(distribution)


confBlank            confHuman            confAnimal           
            mean   min  max      mean  min   max       mean  min   max
group                                                                 
animal  0.397944  0.02  1.0  0.047200  0.0  0.97   0.578603  0.0  0.98
blank   0.940978  0.03  1.0  0.018744  0.0  0.97   0.042679  0.0  0.96
human   0.347676  0.03  1.0  0.646264  0.0  0.97   0.027694  0.0  0.96

CPU times: user 138 ms, sys: 13.8 ms, total: 152 ms
Wall time: 160 ms


In [8]:
# Grab results where aiResults > 0.75 vs total
MIN_CONF_ANIMAL = 0.75
MAX_CONF_ANIMAL = 1.00

def fetch_all_obs():
    query = {"aiResults.0.confAnimal": {"$gte": MIN_CONF_ANIMAL, "$lte": MAX_CONF_ANIMAL}}  # Between
    projection = {
        "_id": 0,
        "mediaID": 1,
        "publicURL": 1,
        "speciesConsensus": 1,
        "aiResults": 1
    }
    all_obs = list(collection.find(query, projection))
    print(f"Retrieved {len(all_obs)} documents with aiResults.confAnimal > {MIN_CONF_ANIMAL} and < {MAX_CONF_ANIMAL}.")
    return all_obs

# Try the fetch operation
try:
    print("===== Starting MongoDB Fetch =====")
    filtered_obs_json = fetch_all_obs()
except Exception as e:
    print(f"Error during fetch: {e}")

===== Starting MongoDB Fetch =====
Retrieved 23611 documents with aiResults.confAnimal > 0.75 and < 1.0.


In [9]:
filtered_obs_json[:2]

[{'mediaID': '9a6f3bbe7d62565c2ce5b632c0dfad55',
  'publicURL': 'https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0160.JPG',
  'speciesConsensus': [{'observationType': 'animal',
    'scientificName': 'Canis familiaris',
    'taxonID': 'Canis familiaris',
    'count': 1,
    'accepted': False,
    'observationCount': 1,
    '_id': ObjectId('67b5ee13b1932f1414ca7ee3')},
   {'observationType': 'human',
    'scientificName': None,
    'taxonID': None,
    'count': 1,
    'accepted': False,
    'observationCount': 2,
    '_id': ObjectId('67b5ee13b1932f1414ca7ee4')}],
  'aiResults': [{'modelName': 'speciesnet/PyTorch/v4.0.1a',
    'runDate': '2025-05-24',
    'confBlank': 0.08,
    'confHuman': 0.92,
    'confAnimal': 0.89}]},
 {'mediaID': 'c8bc6b3e4f8859a06ae30bf269682a27',
  'publicURL': 'https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-31_LearningPlatformBeaver/DCIM/100MEDIA/SYFW0075.JPG',
  'speciesConsensus': [{'observationTyp

In [10]:
# Get the max observations count for any media id
filtered_df = pd.DataFrame(filtered_obs_json)

filtered_df['maxObservationCount'] = filtered_df['speciesConsensus'].apply(
    lambda obs_list: max((item.get('observationCount', 0) for item in obs_list), default=0)
    if isinstance(obs_list, list) else 0
)

with pd.option_context('display.width', 0, 'display.max_colwidth', None):
    display(filtered_df.head(1))

,mediaID,publicURL,speciesConsensus,aiResults,maxObservationCount
0,9a6f3bbe7d62565c2ce5b632c0dfad55,https://urbanriverrangers.s3.amazonaws.com/images/2024/2024-01-30_prologis_02/DCIM/100MEDIA/SYFW0160.JPG,"[{'observationType': 'animal', 'scientificName': 'Canis familiaris', 'taxonID': 'Canis familiaris', 'count': 1, 'accepted': False, 'observationCount': 1, '_id': 67b5ee13b1932f1414ca7ee3}, {'observationType': 'human', 'scientificName': None, 'taxonID': None, 'count': 1, 'accepted': False, 'observationCount': 2, '_id': 67b5ee13b1932f1414ca7ee4}]","[{'modelName': 'speciesnet/PyTorch/v4.0.1a', 'runDate': '2025-05-24', 'confBlank': 0.08, 'confHuman': 0.92, 'confAnimal': 0.89}]",2


In [11]:
# Get count and percent of items with 
sc_count = len(filtered_df['speciesConsensus'].dropna())
total_count = len(filtered_df)

print(f'Total where confAnimal between {MIN_CONF_ANIMAL} to {MAX_CONF_ANIMAL}: \n{total_count} results')

print(f'\nN results where confAnimal between {MIN_CONF_ANIMAL} to {MAX_CONF_ANIMAL} where at least one specesConsensus exists: \n{sc_count} labels')

Total where confAnimal between 0.75 to 1.0: 
23611 results

N results where confAnimal between 0.75 to 1.0 where at least one specesConsensus exists: 
7683 labels


## Current Bracket Labeling Completion Rate:

In [12]:
print("Where at least one speciesConsensus exists...")
print(f'{round(sc_count/total_count*100,2)}% of images are labeled with {MIN_CONF_ANIMAL} < confAnimal < {MAX_CONF_ANIMAL}')

Where at least one speciesConsensus exists...
32.54% of images are labeled with 0.75 < confAnimal < 1.0


In [13]:
# Count where at least 3 people have labeled *something* consistently
## This could still be errors or incomplete labels
sc_count_min3 = (filtered_df['maxObservationCount'] >= 3).sum()

print("Where at least one speciesConsensus exists with at least 3 observations ...")
print(f'{round(sc_count_min3/total_count*100,2)}% of images are labeled with {MIN_CONF_ANIMAL} < confAnimal < {MAX_CONF_ANIMAL}')

Where at least one speciesConsensus exists with at least 3 observations ...
4.48% of images are labeled with 0.75 < confAnimal < 1.0
